<a href="https://colab.research.google.com/github/efitznm/Capstone-Colab/blob/main/Push_PIT_HIC_excels_to_HF_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import io
import pandas as pd
import requests
from google.colab import userdata
import os

In [3]:
# Install pyxlsb to read the .xlsb file format
!pip install -q pyxlsb

In [4]:
# URLs for the data sources
pit_url = "https://www.huduser.gov/portal/sites/default/files/xls/2007-2025-PIT-Counts-by-CoC.xlsb"
hic_url = "https://www.huduser.gov/portal/sites/default/files/xls/2007-2025-HIC-Counts-by-CoC.xlsx"

In [5]:
# Add a browser User-Agent header so huduser.gov allows the download
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        " (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}

In [6]:
# Fetch both files
pit_response = requests.get(pit_url, headers=headers)
pit_response.raise_for_status()  # Raises an error if the download fails

hic_response = requests.get(hic_url, headers=headers)
hic_response.raise_for_status()

In [7]:
# Check if 'PIT data' successful
print(f"""
{'='*40}
Checking PIT file
{'='*40}
Status: {pit_response.status_code}
Content-Type: {pit_response.headers.get("Content-Type")}
Content size: {len(pit_response.content)}
""")


Checking PIT file
Status: 200
Content-Type: application/vnd.ms-excel.sheet.binary.macroEnabled.12
Content size: 8859426



In [8]:
# Check if 'HIC data' successful
print(f"""
{'='*40}
Checking HIC file
{'='*40}
Status: {hic_response.status_code}
Content-Type: {hic_response.headers.get("Content-Type")}
Content size: {len(hic_response.content)}
""")


Checking HIC file
Status: 200
Content-Type: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
Content size: 2598803



In [9]:
# Load the Excel workbooks from memory
pit_excel = pd.ExcelFile(io.BytesIO(pit_response.content), engine="pyxlsb")
hic_excel = pd.ExcelFile(io.BytesIO(hic_response.content))

print("PIT tabs:", pit_excel.sheet_names)
print("HIC tabs:", hic_excel.sheet_names)

PIT tabs: ['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', 'Table_categories', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007', 'CoC Mergers', 'CoCs, States, DDs', 'field list, all years', 'Generate_Table', 'PIT_Counts_Table_Template', 'PIT_Counts_Table', 'Generate_Chart', 'Chart_categories', 'PIT_Counts_Chart', 'PIT_Counts_Chart_Template']
HIC tabs: ['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007', 'Revisions']


In [10]:
# Save downloaded response bytes to local files in Colab
pit_file = "2007-2025-PIT-Counts-by-CoC.xlsb"
hic_file = "2007-2025-HIC-Counts-by-CoC.xlsx"

with open(pit_file, "wb") as f:
    f.write(pit_response.content)

with open(hic_file, "wb") as f:
    f.write(hic_response.content)

!ls -lh {pit_file} {hic_file}

-rw-r--r-- 1 root root 2.5M Nov 21 17:35 2007-2025-HIC-Counts-by-CoC.xlsx
-rw-r--r-- 1 root root 8.5M Nov 21 17:35 2007-2025-PIT-Counts-by-CoC.xlsb


### Authenticate to Hugging Face


In [12]:
# Retrieve the token from Colab's secret manager
os.environ["HF_TOKEN"] = userdata.get('hf_token')
_ = os.environ["HF_TOKEN"]
f"{_[:3]} ... {_[-4:]}"

'hf_ ... RmJe'

In [13]:
os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account = os.environ["HF_ACCOUNT"]
hf_account


'ericbdf7'

In [14]:
hf_repo = "DDDS_Capstone_Housing"
os.environ["HF_REPO"] = hf_repo
hf_repo


'DDDS_Capstone_Housing'

In [15]:
# Log in automatically without an interactive prompt
! hf auth login --token $HF_TOKEN


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Push to Hugging Face


In [18]:
%%capture hf_upload
%%bash

hf upload \
  --type dataset \
  ${HF_ACCOUNT}/${HF_REPO} \
  2007-2025-PIT-Counts-by-CoC.xlsb


In [19]:
print(hf_upload.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/ericbdf7/DDDS_Capstone_Housing/commit/127dbdfd387338563130bd20fda2eeacf89810b4



In [20]:
%%capture hf_upload
%%bash

hf upload \
  --type dataset \
  ${HF_ACCOUNT}/${HF_REPO} \
  2007-2025-HIC-Counts-by-CoC.xlsx


In [21]:
print(hf_upload.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/ericbdf7/DDDS_Capstone_Housing/commit/44bd385bce4a561c7b8e97aea4599a5c01b0c228



# Convert to Parqet and Push to Hugging Face

## Load Both Excel Workbooks from Hugging Face

In [22]:
base_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main"

pit_hf_url = f"{base_url}/2007-2025-PIT-Counts-by-CoC.xlsb"
hic_hf_url = f"{base_url}/2007-2025-HIC-Counts-by-CoC.xlsx"

# Load the workbooks directly from Hugging Face
pit_excel = pd.ExcelFile(pit_hf_url, engine="pyxlsb")
hic_excel = pd.ExcelFile(hic_hf_url)

# Filter to keep ONLY the 4-digit year tabs (2007–2025)
pit_years = [s for s in pit_excel.sheet_names if s.isdigit()]
hic_years = [s for s in hic_excel.sheet_names if s.isdigit()]

print("PIT year sheets:", pit_years)
print("HIC year sheets:", hic_years)

PIT year sheets: ['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007']
HIC year sheets: ['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007']


In [23]:
os.makedirs("pit_parquet", exist_ok=True)
os.makedirs("hic_parquet", exist_ok=True)

def sheets_to_parquet(excel_file, year_sheets, prefix, out_dir):
    for year in year_sheets:
        df = excel_file.parse(year)

        # Clean column names and convert mixed-type 'object' columns to string
        # so PyArrow doesn't fail on suppressed values (like '*' or 'N/A')
        df.columns = df.columns.astype(str)
        for col in df.select_dtypes(include=["object"]).columns:
            df[col] = df[col].astype(str)

        out_path = f"{out_dir}/{prefix}_{year}.parquet"
        df.to_parquet(out_path, index=False)
        print(f"Saved {out_path} ({df.shape[0]} rows, {df.shape[1]} cols)")

sheets_to_parquet(pit_excel, pit_years, "PIT", "pit_parquet")
sheets_to_parquet(hic_excel, hic_years, "HIC", "hic_parquet")

Saved pit_parquet/PIT_2025.parquet (389 rows, 1019 cols)
Saved pit_parquet/PIT_2024.parquet (390 rows, 1018 cols)
Saved pit_parquet/PIT_2023.parquet (389 rows, 474 cols)
Saved pit_parquet/PIT_2022.parquet (390 rows, 406 cols)
Saved pit_parquet/PIT_2021.parquet (390 rows, 406 cols)
Saved pit_parquet/PIT_2020.parquet (395 rows, 406 cols)
Saved pit_parquet/PIT_2019.parquet (400 rows, 406 cols)
Saved pit_parquet/PIT_2018.parquet (399 rows, 406 cols)
Saved pit_parquet/PIT_2017.parquet (398 rows, 406 cols)
Saved pit_parquet/PIT_2016.parquet (397 rows, 315 cols)
Saved pit_parquet/PIT_2015.parquet (398 rows, 316 cols)
Saved pit_parquet/PIT_2014.parquet (394 rows, 91 cols)
Saved pit_parquet/PIT_2013.parquet (395 rows, 91 cols)
Saved pit_parquet/PIT_2012.parquet (394 rows, 47 cols)
Saved pit_parquet/PIT_2011.parquet (395 rows, 47 cols)
Saved pit_parquet/PIT_2010.parquet (394 rows, 31 cols)
Saved pit_parquet/PIT_2009.parquet (394 rows, 27 cols)
Saved pit_parquet/PIT_2008.parquet (393 rows, 25 col

/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


Saved hic_parquet/HIC_2013.parquet (422 rows, 58 cols)


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


Saved hic_parquet/HIC_2012.parquet (429 rows, 48 cols)
Saved hic_parquet/HIC_2011.parquet (434 rows, 43 cols)


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


Saved hic_parquet/HIC_2010.parquet (448 rows, 43 cols)


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


Saved hic_parquet/HIC_2009.parquet (449 rows, 43 cols)
Saved hic_parquet/HIC_2008.parquet (450 rows, 43 cols)
Saved hic_parquet/HIC_2007.parquet (463 rows, 36 cols)


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [24]:
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
repo_id = f"{hf_account}/{hf_repo}"

# Uploads all 19 PIT parquet files into a 'pit_parquet/' subfolder in your repo
api.upload_folder(
    folder_path="pit_parquet",
    path_in_repo="pit_parquet",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Add 2007-2025 PIT yearly parquet files"
)

# Uploads all 19 HIC parquet files into a 'hic_parquet/' subfolder in your repo
api.upload_folder(
    folder_path="hic_parquet",
    path_in_repo="hic_parquet",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Add 2007-2025 HIC yearly parquet files"
)

CommitInfo(commit_url='https://huggingface.co/datasets/ericbdf7/DDDS_Capstone_Housing/commit/c7c5d5119c3f78c8311b2bd758d1e752a86c2ce4', commit_message='Add 2007-2025 HIC yearly parquet files', commit_description='', oid='c7c5d5119c3f78c8311b2bd758d1e752a86c2ce4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ericbdf7/DDDS_Capstone_Housing', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ericbdf7/DDDS_Capstone_Housing'), pr_revision=None, pr_num=None)

## Verify Push to Hugging Face

### Read PIT parquet from Hugging Face


In [35]:
# Construct the base Hugging Face URL for your dataset files
# Point to a single year of PIT Parquet data in your Hugging Face dataset
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/pit_parquet/PIT_2025.parquet"
hf_url


'https://huggingface.co/datasets/ericbdf7/DDDS_Capstone_Housing/resolve/main/pit_parquet/PIT_2025.parquet'

In [36]:
df3 = pd.read_parquet( hf_url )
df3.shape


(389, 1019)

In [37]:
df3.iloc[:,:5].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 389 entries, 0 to 388
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CoC Number        389 non-null    object 
 1   CoC Name          389 non-null    object 
 2   CoC Category      389 non-null    object 
 3   Count Types       389 non-null    object 
 4   Overall Homeless  387 non-null    float64
dtypes: float64(1), object(4)
memory usage: 15.3+ KB


### Read HIC parquet from Hugging Face


In [38]:
# Construct the base Hugging Face URL for your dataset files
# Point to a single year of PIT Parquet data in your Hugging Face dataset
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/hic_parquet/HIC_2025.parquet"
hf_url


'https://huggingface.co/datasets/ericbdf7/DDDS_Capstone_Housing/resolve/main/hic_parquet/HIC_2025.parquet'

In [39]:
df3 = pd.read_parquet( hf_url )
df3.shape


(387, 77)

In [40]:
df3.iloc[:,:5].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 387 entries, 0 to 386
Data columns (total 5 columns):
 #   Column                                                    Non-Null Count  Dtype  
---  ------                                                    --------------  -----  
 0   CoC Number                                                387 non-null    object 
 1   Total Year-Round Beds (ES, TH, SH)                        387 non-null    int64  
 2   Total Non-DV Year-Round Beds (ES, TH, SH)                 387 non-null    int64  
 3   Total HMIS Year-Round Beds (ES, TH, SH)                   387 non-null    int64  
 4   HMIS Participation Rate for Year-Round Beds (ES, TH, SH)  387 non-null    float64
dtypes: float64(1), int64(3), object(1)
memory usage: 15.2+ KB
